In [1]:
import geopandas as gpd

# Load the local file you just downloaded
gdf = gpd.read_file("local_greenhouses.geojson")

print(f"Success! Securely loaded {len(gdf)} building records into memory in milliseconds.")
print(gdf[['id', 'geometry']].head())

Success! Securely loaded 13 building records into memory in milliseconds.
                                     id  \
0  0eac054c-c24e-4339-a2f5-daf00385de64   
1  af11f45d-9a32-4151-8450-a4d62b1cbb86   
2  a21afdde-844b-4c06-9c81-37ee02ab0aab   
3  1631aeb9-d7aa-46a8-acc1-57ead568f223   
4  2508b4dd-a42d-4bba-95db-48b5c07046ad   

                                            geometry  
0  POLYGON ((-69.80011 9.97366, -69.80013 9.97361...  
1  POLYGON ((-69.79887 9.97207, -69.79898 9.97117...  
2  POLYGON ((-69.79861 9.97166, -69.79868 9.97113...  
3  POLYGON ((-69.79821 9.97195, -69.79833 9.97106...  
4  POLYGON ((-69.79897 9.97217, -69.79916 9.97218...  


In [ ]:
import duckdb
from lonboard import Map, SolidPolygonLayer

# 1. Start DuckDB connection and fully parallelize engine configurations
con = duckdb.connect()
con.execute("SET threads = 8;        -- Forces concurrent parsing loops")
con.execute("SET max_memory = '8GB'; -- Expands operational processing buffer")

# 2. Grab the live release version token dynamically
latest_release = con.execute("""
    SELECT latest FROM read_json('https://stac.overturemaps.org/catalog.json')
""").fetchone()[0]

print(f"Streaming live dataset from Release: {latest_release}")

# Your exact 1 km target box coordinates (Long, Lat)
xmin, ymin, xmax, ymax = -69.8040, 9.9660, -69.7950, 9.9750

# 3. ULTRALIGHT ENGINE QUERY: Filter directly using Overture's native structural box columns
# This forces the cloud metadata engine to discard non-matching file parts instantly.
query = f"""
    SELECT 
        id, 
        geometry
    FROM read_parquet(
        'az://overturemapswestus2.blob.core.windows.net/release/{latest_release}/theme=buildings/type=building/*.parquet', 
        hive_partitioning=1
    )
    WHERE bbox.xmin >= {xmin} AND bbox.xmax <= {xmax}
      AND bbox.ymin >= {ymin} AND bbox.ymax <= {ymax}
"""

print("Executing parallel index filter query...")
# .arrow() Streams data blocks natively directly into RAM without format mutations
arrow_table = con.execute(query).arrow()

if arrow_table.num_rows == 0:
    print("\nNo building geometries matched this exact 1 km geographic space.")
else:
    print(f"\nSuccess! Pulled {arrow_table.num_rows} records seamlessly.")
    
    # 4. GPU-Accelerated Visualization
    layer = SolidPolygonLayer.from_arrow(arrow_table)
    m = Map(layers=[layer])
    display(m)

Streaming live dataset from Release: 2026-05-20.0
Executing parallel index filter query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))